In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

GoogleGeminiKey = os.getenv('GOOGLE_GEMINI_KEY')
connection = os.getenv("PG_CONNECTION_STRING")


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_postgres.vectorstores import PGVector

raw_documents = TextLoader("data/how_to_read_pnl.txt").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GoogleGeminiKey)
db = PGVector.from_documents(documents, embeddings, connection=connection)

In [14]:
# create retriever
retriever = db.as_retriever(search_kwargs={"k":10})

# fetch relevant documents.
docs = retriever.invoke("how to read pnl?")
docs

[Document(id='7f623b30-63c0-40a2-a295-a85513791a1c', metadata={'source': 'data/how_to_read_pnl.txt'}, page_content='Tata McGraw Hill Publishing Company Limited\nNEW DELHI\nN. Ramachandran\nPrincipal Consultant\nManagement Advisory Services\nKochi\nRam Kumar Kakani\nAssociate Professor, XLRI\nJamshedpur\nHow to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw Hill Professional: Finance Made Easy Series\nPublished by Tata McGraw Hill Education Private Limited,\n7 West Patel Nagar, New Delhi 110 008.\nCopyright © 2010, by Tata McGraw Hill Education Private Limited\nNo part of this publication may be reproduced or distributed in any form or by any\nmeans, electronic, mechanical, photocopying, recording, or otherwise or stored in a\ndatabase or retrieval system without the prior written permission of the publishers. The\nprogram listings (if any) may be entered, stored and executed in a computer system,\nbut they may not be reproduced for publication.\nThis edition can be exported from Indi

In [23]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Markdown, display

llm = GoogleGenerativeAI(model="gemini-2.0-flash-001",api_key=GoogleGeminiKey,temperature=0)

prompt_template = ChatPromptTemplate.from_messages(
[("system", """
            Answer the question on the based on the given context below. If the question cannot be answered using the information
provided in the context, respond with "I don't know".
    """),
    ("system","Answer always in markdown format and List the items."),
    ("human", "Context: {context}"),
    ("human", "Question: {question}")])

chain = prompt_template | llm

query = "how to read pnl?"

docs = retriever.get_relevant_documents(query)

result = chain.invoke({"context": docs, "question": query})

display(Markdown(result))


To effectively read a Profit and Loss (P&L) statement, here are key steps and components to consider:

1.  **Understand the basic structure:**
    *   Sales (or Revenues)
    *   Less: Cost of Goods Sold
    *   = Gross Profit
    *   Less: Operating Expenses
    *   = Operating Profit
    *   Less: Non-Operating Expenses
    *   = Profit Before Interest and Tax (PBIT)
    *   Less: Interest
    *   = Profit Before Tax (PBT)
    *   Less: Tax
    *   = Profit After Tax (PAT)
    *   Add: Previous year’s balance of P&L A/C
    *   = Profit Available for Distribution
    *   Less: Appropriations
    *   = Retained Earnings

2.  **Key Profit Metrics:**
    *   **Gross Profit:**  Indicates the profit a company makes after deducting the costs associated with producing and selling its products or services.
    *   **Operating Profit:**  Reveals the profit from core business operations before interest and taxes.
    *   **PBIT (Profit Before Interest and Tax):**  Earnings before accounting for interest expenses and taxes.
    *   **PBT (Profit Before Tax):**  Earnings before taxes.
    *   **PAT (Profit After Tax):**  The net income earned by a company after providing for tax. It measures the net earnings available to the shareholders.

3.  **Profit Available for Distribution:**
    *   Sum up PAT and the retained earnings from previous years.
    *   This figure indicates the amount available for distribution, such as dividends.

4.  **Retained Earnings:**
    *   Derived by deducting the dividend to be paid to shareholders from the profit available for distribution.

5.  **Non-Operating Income:**
    *   Income from activities that are not part of the core business.

6.  **Depreciation Methods:**
    *   **Straight Line/Fixed Method:** A fixed percentage of the original cost of the asset is depreciated each year until the asset value becomes nil.
    *   **Written Down Value Method:** Also referred to as the remaining book value, this is the value at which assets are recorded on the balance sheet.

7.  **Importance of Income Statement:**
    *   Shows the change in owner's equity affected by retained earnings.
    *   Provides knowledge about the operations of the business.